<a href="https://colab.research.google.com/github/haddybhaiya/flowState/blob/main/train_gw_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')




Mounted at /content/drive


In [2]:
import os

base = "/content/drive/MyDrive/flowState"

os.makedirs(base + "/data/processed", exist_ok=True)
os.makedirs(base + "/models/lstm", exist_ok=True)


In [3]:
!pip install tensorflow pandas numpy scikit-learn matplotlib joblib



In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping


In [74]:
well = "Balaji_nagar"
path = f"{base}/data/processed/well_{well}.csv"

df = pd.read_csv(path)
df.head()


,\ndate,gwl,rain,rain_lag_7,rain_lag_30,cum_rain_30,gwl_lag_1,gwl_trend_30,month_sin,month_cos
0,2023-03-17,-88.089996,0,0.0,0.0,0.0,-88.089996,0.0,1.0,6.123234e-17
1,2023-03-18,-88.089996,0,0.0,0.0,0.0,-88.089996,0.0,1.0,6.123234e-17
2,2023-03-19,-88.089996,0,0.0,0.0,0.0,-88.089996,0.0,1.0,6.123234e-17
3,2023-03-20,-88.089996,0,0.0,0.0,0.0,-88.089996,0.0,1.0,6.123234e-17
4,2023-03-21,-88.089996,0,0.0,0.0,0.0,-88.089996,0.0,1.0,6.123234e-17


In [75]:
features = [
    "rain", "rain_lag_7", "rain_lag_30",
    "cum_rain_30", "gwl_lag_1",
    "gwl_trend_30", "month_sin", "month_cos"
]

X = df[features].values
y = df["gwl"].values.reshape(-1, 1)


In [76]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)


In [77]:
SEQ_LEN = 30

X_seq, y_seq = [], []
for i in range(len(X_scaled) - SEQ_LEN):
    X_seq.append(X_scaled[i:i+SEQ_LEN])
    y_seq.append(y_scaled[i+SEQ_LEN])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)


In [78]:
split = int(0.8 * len(X_seq))
X_train, X_val = X_seq[:split], X_seq[split:]
y_train, y_val = y_seq[:split], y_seq[split:]


In [79]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, X_seq.shape[2])),
    LSTM(32),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_14 (LSTM)                  │ (None, 30, 64)         │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_15 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,137 (121.63 KB)

 Trainable params: 31,137 (121.63 KB)

 Non-trainable params: 0 (0.00 B)

In [80]:
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[EarlyStopping(patience=5)],
    verbose=1
)


Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - loss: 0.0034 - val_loss: 0.0442
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0064 - val_loss: 0.0189
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0046 - val_loss: 0.0122
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0015 - val_loss: 0.0092
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0086 - val_loss: 0.0075
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0054 - val_loss: 0.0047
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0029 - val_loss: 0.0052
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0035 - val_loss: 0.0048
Epoch 9/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0020 - val_loss: 0.0081
Epoch 10/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0016 - val_loss: 0.0100
Epoch 11/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0018 - val_loss: 0.0067


In [81]:
model.save(f"{base}/models/lstm/{well}.keras")


In [82]:
import joblib

joblib.dump(scaler_X, f"{base}/models/lstm/{well}_scaler_X.pkl")
joblib.dump(scaler_y, f"{base}/models/lstm/{well}_scaler_y.pkl")


['/content/drive/MyDrive/flowState/models/lstm/Balaji_nagar_scaler_y.pkl']

In [83]:
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score

# Make predictions on the validation set
y_pred_scaled = model.predict(X_val)

# Inverse transform the scaled predictions and actual values to their original scale
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = scaler_y.inverse_transform(y_val)

# Calculate evaluation metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
r2_score(y_true, y_pred)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Mean Absolute Error (MAE): 1.6570
Root Mean Squared Error (RMSE): 1.7989


0.2575641967200908